# Certilab Adaptive RAG — Demo en vivo

Este notebook ejecuta el grafo Adaptive RAG canónico paso a paso usando `graph.stream()`. Requiere `OPENAI_API_KEY` configurada en el entorno.

**Referencia**: [Building an Adaptive RAG System with LangGraph, OpenAI and Tavily](https://levelup.gitconnected.com/building-an-adaptive-rag-system-with-langgraph-openai-and-tavily-c4ee39d2f021)

In [ ]:
import os
assert "OPENAI_API_KEY" in os.environ, "Configura OPENAI_API_KEY antes de ejecutar este notebook"

In [ ]:
from pathlib import Path

from app.adaptive_rag.graph import build_graph
from app.adaptive_rag.grader import (  # noqa: F401
    RouteQuery,
    GradeDocuments,
    GradeHallucinations,
    GradeAnswer,
    QuestionRewriter,
)
from app.domain.models import Role
from app.ingestion.indexer import InMemoryVectorIndex
from app.ingestion.loader import load_certificates, load_customers, load_histories, load_pdf_texts
from app.ingestion.splitter import build_pdf_chunks
from app.security.access_control import Principal, scope_from_principal
from app.tools.web_search import TavilyWebSearch, WebSearchConfig

In [ ]:
# Cargar datos mock y construir componentes
data_dir = Path("data")
certificates = load_certificates(data_dir)
load_customers(data_dir)
load_histories(data_dir)
pdf_texts = load_pdf_texts(data_dir)
chunks = build_pdf_chunks(certificates, pdf_texts)

index = InMemoryVectorIndex(chunks=chunks)
embeddings = None  # El indice embeddea internamente
web_search = TavilyWebSearch(WebSearchConfig(tavily_api_key=os.environ.get("TAVILY_API_KEY")))

graph = build_graph(index=index, embeddings=embeddings, web_search=web_search)
print(f"Grafo construido con {len(chunks)} chunks indexados.")

In [ ]:
# Query A — Ruta vectorstore estandar
question_A = "Cuantos certificados vigentes tiene el cliente 101?"

principal = Principal(role=Role.ADMIN, customer_id=None, user_id=1)
scope = scope_from_principal(principal)

initial_state_A = {
    "question": question_A,
    "generation": "",
    "documents": [],
    "web_results": [],
    "route": "",
    "rewrite_count": 0,
    "regenerate_count": 0,
    "hallucination_verdict": "",
    "answer_verdict": "",
    "principal": principal,
    "scope": scope,
}

In [ ]:
for step in graph.stream(initial_state_A):
    node_name = list(step.keys())[0]
    print(f"Node: {node_name}")
    print(step)
    print()

## Loop de auto-correccion: Reescritura de query

Cuando los documentos recuperados no son relevantes para la pregunta, el grafo reescribe la consulta y vuelve a intentar la recuperacion.

In [ ]:
# Query B — Ambigua, dispara el loop de reescritura
question_B = "Dame info"

initial_state_B = {
    "question": question_B,
    "generation": "",
    "documents": [],
    "web_results": [],
    "route": "",
    "rewrite_count": 0,
    "regenerate_count": 0,
    "hallucination_verdict": "",
    "answer_verdict": "",
    "principal": principal,
    "scope": scope,
}

In [ ]:
for step in graph.stream(initial_state_B):
    node_name = list(step.keys())[0]
    node_output = step[node_name]
    if isinstance(node_output, dict):
        rewrite_count = node_output.get("rewrite_count")
        if rewrite_count is not None:
            print(f"rewrite_count ahora = {rewrite_count}")
    print(f"Node: {node_name}")
    print(step)
    print()

## Loop de auto-correccion: Verificacion de alucinaciones

Despues de generar una respuesta, el grafo verifica que este respaldada por los documentos recuperados.

In [ ]:
# Query C — Puede disparar la verificacion de alucinaciones
question_C = "Que procedimientos de calibracion siguen las normas ISO?"

initial_state_C = {
    "question": question_C,
    "generation": "",
    "documents": [],
    "web_results": [],
    "route": "",
    "rewrite_count": 0,
    "regenerate_count": 0,
    "hallucination_verdict": "",
    "answer_verdict": "",
    "principal": principal,
    "scope": scope,
}

In [ ]:
for step in graph.stream(initial_state_C):
    node_name = list(step.keys())[0]
    node_output = step[node_name]
    if isinstance(node_output, dict):
        verdict = node_output.get("hallucination_verdict")
        regen = node_output.get("regenerate_count")
        if verdict:
            print(f"hallucination_verdict = {verdict}")
        if regen is not None:
            print(f"regenerate_count ahora = {regen}")
    print(f"Node: {node_name}")
    print(step)
    print()

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

## Resumen

- Enrutamiento adaptativo (vectorstore vs. web search)
- Gradeo de relevancia de documentos
- Reescritura de query con loop acotado (max. 3 intentos)
- Verificacion de alucinaciones con loop acotado (max. 2 intentos)
- Roles y scope por cliente preservados